# Samsung Cloud Firmware → TWRP Device Tree

This notebook now bundles the **full twrpdtgen-NEXT workflow** into the cloud path.

It can either:

1. download/decrypt Samsung firmware in Colab and return only one extracted `.img`, or
2. continue directly into twrpdtgen, generate the complete candidate TWRP device tree, ZIP it, and download only that ZIP to your computer.

The full firmware is processed inside Colab and is removed after extraction by default. Advanced retention settings remain available through the CLI.

In [ ]:
import os, subprocess, sys
repo = '/content/twrpdtgen-NEXT'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/RectifyData/twrpdtgen-NEXT.git', repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'pull'], check=True)
os.chdir(repo)
# Use a separate Python process for installation and each generation run.
# An older module imported into this Colab kernel will not be reused.
subprocess.run(['bash', 'launcher.sh', '--setup-only'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
tool_version = subprocess.check_output(
    [sys.executable, '-c', 'import twrpdtgen; print(twrpdtgen.__version__)'],
    text=True,
).strip()
print(f'twrpdtgen-NEXT {tool_version} @ {commit} ready')

## Firmware and output options

Choose **Extract image only** for a standalone `.img`, or **Generate TWRP device tree** for a source-tree ZIP.

`firmware_version` is optional; leave it empty to request the latest Samsung firmware. It is **not** the twrpdtgen tool version.

The standard AP package contains Android boot/recovery images. For most devices, keep `recovery_layout = auto`. No unverified partition-size fields or unused download switches are required.

In [ ]:
workflow = 'Extract image only' #@param ['Extract image only', 'Generate TWRP device tree']
model = 'SM-A207F' #@param {type:'string'}
region = 'NEE' #@param {type:'string'}
partition = 'recovery' #@param {type:'string'}
firmware_version = '' #@param {type:'string'}
package = 'AP'
threads = 8
skeleton = 'auto' #@param ['auto', 'omni', 'twrp-12.1', 'twrp-14.1']
recovery_layout = 'auto' #@param ['auto', 'dedicated', 'boot', 'vendor_boot']
recovery_root_mode = 'minimal' #@param ['minimal']

In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import files

if not model.strip() or not region.strip() or not partition.strip():
    raise ValueError('Model, region and partition are required.')

name = Path(partition.strip()).name
if name.lower().endswith('.lz4'):
    name = name[:-4]
if not name.lower().endswith(('.img', '.bin')):
    name += '.img'

image_output = Path('/content') / name
tree_output = Path('/content/twrp-tree')
tree_archive = Path('/content/twrp-device-tree.zip')

if workflow == 'Extract image only':
    image_output.unlink(missing_ok=True)
    args = [
        'fetch', '-m', model, '-r', region,
        '--partition', partition,
        '--package', package,
        '-j', str(threads),
        '-o', str(image_output),
        '--overwrite',
    ]
    if firmware_version.strip():
        args.extend(['-v', firmware_version.strip()])
    cmd = [sys.executable, '-m', 'twrpdtgen.cloud_slice', *args]
    print('Extracting image:', name)
    subprocess.run(cmd, check=True)
    if not image_output.is_file():
        raise FileNotFoundError(f'Extraction finished without {image_output}')
    files.download(str(image_output))

elif workflow == 'Generate TWRP device tree':
    tree_archive.unlink(missing_ok=True)
    args = [
        '-m', model, '-r', region,
        '--partition', partition,
        '--package', package,
        '-j', str(threads),
        '--tree-output', str(tree_output),
        '--archive', str(tree_archive),
        '--skeleton', skeleton,
        '--recovery-layout', recovery_layout,
        '--recovery-root-mode', recovery_root_mode,
        '--overwrite',
    ]
    if firmware_version.strip():
        args.extend(['-v', firmware_version.strip()])
    cmd = [sys.executable, '-m', 'twrpdtgen.cloud_tree', *args]
    print('Generating TWRP tree:', name)
    subprocess.run(cmd, check=True)
    if not tree_archive.is_file():
        raise FileNotFoundError(f'Generation finished without {tree_archive}')
    files.download(str(tree_archive))

else:
    raise ValueError(f'Unknown workflow: {workflow!r}')

## Results

**Extract image only:** download the selected `.img` file.

**Generate TWRP device tree:** download `twrp-device-tree.zip`. The expanded tree also remains in `/content/twrp-tree/` for inspection until the Colab runtime ends.

The ZIP is a **candidate source tree**, not a boot-verified recovery. Check partition geometry, kernel, AVB, SELinux, encryption and real hardware compatibility before flashing.

The archive excludes `recovery/stock-root/` by default because the original ramdisk is only needed for auditing and can be very large. The expanded stock-root remains in Colab. Symlinks are preserved in the ZIP and require a symlink-capable extractor.

Generation and extraction run in new Python subprocesses, so a Colab kernel won't mix old imported twrpdtgen code with newly pulled sources.